# 2D Inverse FFT Interactive Demo

任意の画像が正弦波の重ね合わせであることをインタラクティブに学ぶデモです。

## 起動手順（Google Colab）

1. **画像の準備セル**で `GDRIVE_IMAGE_DIR` を自分の Drive パスに変更して実行する
2. Google Drive のマウント許可ダイアログで「許可」を選ぶ
3. 残りのセルをすべて順に実行する

## 起動手順（ローカル Jupyter）

1. **画像の準備セル**で `IMAGE_SOURCE = 'local'` に変更して実行する
2. 残りのセルをすべて順に実行する

## 操作方法

- 振幅スペクトル（上中・下中パネル）の上でマウスをドラッグする
- 通過した周波数成分が逐次 IFFT に加算され、**下左パネル**に再合成画像が現れる
- **下右パネル**に最後に追加したグレーティング（正弦波成分）が表示される
- **Reset** ボタンでマスクをリセット

---
Python port of: Sasaki & Ohzawa (2009), Osaka University — BSD License
本デモはKota S. Sasaki, Izumi Ohzawa両氏による"Two-Dimensional Fourier Image Reconstruction (Inverse FT) Demo using Matlab"をPythonに移植したものです。

Two-Dimensional Fourier Image Reconstruction (Inverse FT) Demo using Matlab

Authors: Kota S. Sasaki and Izumi Ohzawa
Graduate School of Frontier Biosciences, Osaka University
kota@fbs.osaka-u.ac.jp, ohzawa@fbs.osaka-u.ac.jp

License: BSD license, http://creativecommons.org/licenses/BSD/

Modification Date: 2009-05-01

Modification Date: 2016-08-19
     IO - added 2 more images.
          Near Line 184　removed "‾isnan(s.hCurrentImage) &" for fix for newer Matlab.
		  (Since R2014b, image handle type became object from double?)

Modification Date: 2026-07-16
     Akira ITOH - Ported the original MATLAB implementation to Python.


In [ ]:
# =================================================================
# ステップ 1: ipympl インストール確認
# =================================================================
# 【Colab】このセルを実行 → 「再起動が必要です」と出たらランタイムを再起動
#          → 再起動後、このセルを再実行 → 「準備完了」と出たら次へ進む
# 【ローカル Jupyter】pip install ipympl を実行済みなら「準備完了」と出る
# =================================================================
import importlib, sys

if importlib.util.find_spec('ipympl') is None:
    print("ipympl が見つかりません。インストールします...")
    import subprocess
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', 'ipympl'],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print("インストールしました。")
    else:
        print("インストール失敗:", result.stderr)

    print()
    print("=" * 55)
    print("【重要】ランタイムの再起動が必要です")
    print("  Colab メニュー: ランタイム → セッションを再起動")
    print("  再起動後、このセル（ステップ1）を再実行してください")
    print("=" * 55)
else:
    # インストール済み → widget manager を有効化
    if 'google.colab' in sys.modules:
        from google.colab import output
        output.enable_custom_widget_manager()
    print("ipympl 準備完了。ステップ 2 のセルに進んでください。")

In [ ]:
# =================================================================
# ステップ 2: matplotlib バックエンド設定
# =================================================================
# ステップ 1 で「準備完了」と出てからこのセルを実行してください
# =================================================================
import importlib
if importlib.util.find_spec('ipympl') is None:
    raise RuntimeError(
        "ipympl が見つかりません。ステップ 1 のセルに戻り、"
        "インストール → ランタイム再起動 → ステップ 1 再実行 の手順を完了してください。"
    )

%matplotlib widget
print("インタラクティブバックエンド 設定完了。")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.widgets import Button
from PIL import Image
import urllib.request
import os

In [ ]:
# =================================================================
# 画像の準備
# =================================================================
# 【Colab の場合】GDRIVE_IMAGE_DIR を自分の Google Drive パスに変更して実行
# 【ローカル Jupyter の場合】IMAGE_SOURCE = 'local' に変更して実行
# =================================================================

import sys, os
IN_COLAB = 'google.colab' in sys.modules

IMAGE_SOURCE   = 'gdrive'                # 'gdrive' | 'local'
IMAGE_FILENAME = 'TaiyounoTou128x128t.png'
# 他の選択肢: 'TaiyounoTou128x128.png' / 'lena_std128x128.png' / 'Albert_Einstein_Nobel_s.png'

# Google Drive 内の画像フォルダパス（MyDrive 以下のパスを指定）
GDRIVE_SUBDIR = 'Colab_mnt_2DIFFT_demo'  # ← MyDrive 直下のフォルダ名に変更してください

# =================================================================

IMAGE_PATH = None

def _resolve_mydrive():
    """MyDrive のパスを返す（大文字小文字ゆれに対応）"""
    for candidate in ['/content/drive/MyDrive', '/content/drive/Mydrive']:
        if os.path.exists(candidate):
            return candidate
    return None

if IMAGE_SOURCE == 'gdrive':
    if not IN_COLAB:
        raise RuntimeError("IMAGE_SOURCE='gdrive' は Colab 環境専用です。"
                           "ローカル実行の場合は IMAGE_SOURCE='local' に変更してください。")
    from google.colab import drive
    drive.mount('/content/drive')

    mydrive = _resolve_mydrive()
    if mydrive is None:
        raise RuntimeError("Google Drive のマウントに失敗しました。再度セルを実行してください。")

    gdrive_dir = os.path.join(mydrive, GDRIVE_SUBDIR)

    # 指定ファイル名が見つからない場合、フォルダ内の画像一覧を表示して候補を提示
    IMAGE_PATH = os.path.join(gdrive_dir, IMAGE_FILENAME)
    if not os.path.exists(IMAGE_PATH):
        print(f"[ERROR] ファイルが見つかりません: {IMAGE_PATH}\n")
        if os.path.isdir(gdrive_dir):
            imgs = [f for f in os.listdir(gdrive_dir)
                    if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'))]
            if imgs:
                print(f"フォルダ '{gdrive_dir}' 内の画像ファイル:")
                for f in sorted(imgs):
                    print(f"  {f}")
                print(f"\nIMAGE_FILENAME をこれらのいずれかに変更して再実行してください。")
            else:
                print(f"フォルダ '{gdrive_dir}' に画像ファイルが見つかりません。")
                print("画像をそのフォルダにアップロードしてから再実行してください。")
        else:
            print(f"フォルダが存在しません: {gdrive_dir}")
            print(f"GDRIVE_SUBDIR を正しいフォルダ名に変更してください。")
            print(f"\nMyDrive 直下のフォルダ一覧:")
            for entry in sorted(os.listdir(mydrive)):
                if os.path.isdir(os.path.join(mydrive, entry)):
                    print(f"  {entry}/")
        IMAGE_PATH = None

elif IMAGE_SOURCE == 'local':
    _base = os.path.normpath(os.path.join(os.getcwd(), '..'))
    _candidates = [
        os.path.join(_base, 'MATLAB', 'InverseFFT2D', 'images', IMAGE_FILENAME),
        os.path.join('images', IMAGE_FILENAME),
        IMAGE_FILENAME,
    ]
    for _p in _candidates:
        if os.path.exists(_p):
            IMAGE_PATH = os.path.normpath(_p)
            break
    if IMAGE_PATH is None:
        raise FileNotFoundError(
            "画像が見つかりません。以下を確認してください:\n"
            + "\n".join(f"  {os.path.normpath(p)}" for p in _candidates)
        )

else:
    raise ValueError(f"IMAGE_SOURCE='{IMAGE_SOURCE}' は未対応です。'gdrive' か 'local' を指定してください。")

if IMAGE_PATH:
    print(f"使用画像: {IMAGE_PATH}")

In [ ]:
# =================================================================
# [Colab 用] Drive 内のフォルダ・ファイルを確認するヘルパー
# file not found エラーが出た場合にここを実行して正しいパスを探してください
# =================================================================

import os

def list_drive(path='/content/drive/MyDrive', depth=2, _indent=0):
    """Drive 内のディレクトリ構造を表示する"""
    if _indent == 0:
        print(f"{path}/")
    if depth == 0:
        return
    try:
        entries = sorted(os.listdir(path))
    except PermissionError:
        return
    for entry in entries:
        full = os.path.join(path, entry)
        prefix = '  ' * (_indent + 1)
        if os.path.isdir(full):
            print(f"{prefix}{entry}/")
            list_drive(full, depth - 1, _indent + 1)
        else:
            print(f"{prefix}{entry}")

# MyDrive の上位2階層を表示（画像の場所を特定してください）
list_drive('/content/drive/MyDrive', depth=2)

In [ ]:
# -----------------------------------------------------------------
# コア関数
# -----------------------------------------------------------------

def get_luminance_image(filename):
    """画像を読み込みグレースケール輝度に変換（MATLABのGetLuminanceImage相当）"""
    img = Image.open(filename)
    X = np.array(img, dtype=float)
    if X.ndim == 3:
        L = 0.30 * X[:, :, 0] + 0.59 * X[:, :, 1] + 0.11 * X[:, :, 2]
    else:
        L = X
    return L - 127.5


def myff2(X, m=None, n=None):
    """2D FFT + fftshift、周波数軸を返す（MATLABのMyff2相当）"""
    if m is None:
        m, n = X.shape
    Y = np.fft.fftshift(np.fft.fft2(X, s=(m, n)))
    f0 = np.floor(np.array([m, n]) / 2).astype(int) + 1  # 1-indexed center
    # MATLABの fy = ((m:-1:1) - f0(1) + 1) / m  に対応
    fy = (np.arange(m, 0, -1) - f0[0] + 1) / m   # 上→下が高周波→低周波
    fx = (np.arange(1, n + 1) - f0[1]) / n
    return Y, fx, fy


def draw_line_on_mask(mask, xi_new, yi_new, xi_old, yi_old):
    """2点間を補間してマスクに線を描く（MATLABのUpdateMask内の補間ロジック相当）"""
    if xi_old is None:
        mask[yi_new, xi_new] = 1
        return mask

    xi, yi = xi_new, yi_new
    xo, yo = xi_old, yi_old

    if xi == xo:
        yy = np.arange(min(yi, yo), max(yi, yo) + 1)
        xx = np.full_like(yy, xi)
    else:
        slope = (yi - yo) / (xi - xo)
        if abs(slope) < 1:
            xx = np.arange(min(xi, xo), max(xi, xo) + 1)
            yy = np.round(slope * (xx - xi) + yi).astype(int)
        else:
            yy = np.arange(min(yi, yo), max(yi, yo) + 1)
            xx = np.round((1 / slope) * (yy - yi) + xi).astype(int)

    # 範囲クリップ
    rows, cols = mask.shape
    valid = (xx >= 0) & (xx < cols) & (yy >= 0) & (yy < rows)
    mask[yy[valid], xx[valid]] = 1
    return mask

In [ ]:
# -----------------------------------------------------------------
# デモ本体クラス
# -----------------------------------------------------------------

class IFFTDemo:
    MIN_AMP = 1e-10

    def __init__(self, filename):
        self.filename = filename
        self._load_and_init()
        self._build_figure()
        self._connect_events()
        self._update_ifft()

    # ----------------------------------------------------------
    # 初期化
    # ----------------------------------------------------------
    def _load_and_init(self):
        self.L = get_luminance_image(self.filename)
        fft_pts = 2 ** np.ceil(np.log2(self.L.shape)).astype(int)
        self.FFTedL, self.fx, self.fy = myff2(self.L, fft_pts[0], fft_pts[1])
        self.mask = np.zeros(self.FFTedL.shape)

        amp = np.abs(self.FFTedL)
        amp = np.where(amp < self.MIN_AMP, self.MIN_AMP, amp)
        self.amp = np.log10(amp)

        self._dragging = False
        self._old_xi = None
        self._old_yi = None
        self._picking_ax = None

    # ----------------------------------------------------------
    # 図の構築
    # ----------------------------------------------------------
    def _build_figure(self):
        self.fig = plt.figure(figsize=(12, 8))
        self.fig.canvas.header_visible = False

        gs = gridspec.GridSpec(2, 3, figure=self.fig,
                               hspace=0.4, wspace=0.35)

        self.ax_orig   = self.fig.add_subplot(gs[0, 0])
        self.ax_amp    = self.fig.add_subplot(gs[0, 1])
        self.ax_ifft   = self.fig.add_subplot(gs[1, 0])
        self.ax_picker = self.fig.add_subplot(gs[1, 1])
        self.ax_grat   = self.fig.add_subplot(gs[1, 2])

        cmap = 'gray'
        # extent = [left, right, bottom, top] in data coords
        ext = [self.fx[0], self.fx[-1], self.fy[-1], self.fy[0]]

        # 元画像
        self.ax_orig.imshow(self.L, cmap=cmap, origin='upper', aspect='equal')
        self.ax_orig.set_title('original image')
        self.ax_orig.set_xlabel('x')
        self.ax_orig.set_ylabel('y')

        # 振幅スペクトル（クリック受付）
        self.im_amp = self.ax_amp.imshow(
            self.amp, cmap=cmap, origin='upper', extent=ext, aspect='equal')
        self.ax_amp.set_title('amplitude spectrum')
        self.ax_amp.set_xlabel('fx (cyc/pix)')
        self.ax_amp.set_ylabel('fy (cyc/pix)')

        # IFFTed image
        blank = np.zeros(self.L.shape)
        self.im_ifft = self.ax_ifft.imshow(
            blank, cmap=cmap, origin='upper', aspect='equal')
        self.ax_ifft.set_title('IFFTed image')
        self.ax_ifft.set_xlabel('x')
        self.ax_ifft.set_ylabel('y')

        # Picker（選択済みスペクトル）
        self.im_picker = self.ax_picker.imshow(
            np.zeros_like(self.amp), cmap=cmap, origin='upper',
            extent=ext, aspect='equal')
        self.ax_picker.set_title('unmasked amplitude spectrum')
        self.ax_picker.set_xlabel('fx (cyc/pix)')
        self.ax_picker.set_ylabel('fy (cyc/pix)')

        # 最後のグレーティング
        self.im_grat = self.ax_grat.imshow(
            blank, cmap=cmap, origin='upper', aspect='equal')
        self.ax_grat.set_title('most recent grating added')
        self.ax_grat.set_xlabel('x')
        self.ax_grat.set_ylabel('y')

        # Reset ボタン
        ax_btn = self.fig.add_axes([0.78, 0.92, 0.1, 0.04])
        self.btn_reset = Button(ax_btn, 'Reset')
        self.btn_reset.on_clicked(self._on_reset)

        self.fig.canvas.draw()

    # ----------------------------------------------------------
    # イベント接続
    # ----------------------------------------------------------
    def _connect_events(self):
        c = self.fig.canvas
        c.mpl_connect('button_press_event',   self._on_press)
        c.mpl_connect('motion_notify_event',  self._on_motion)
        c.mpl_connect('button_release_event', self._on_release)

    def _is_picker_ax(self, ax):
        return ax in (self.ax_amp, self.ax_picker)

    # ----------------------------------------------------------
    # マウスイベントハンドラ
    # ----------------------------------------------------------
    def _on_press(self, event):
        if event.inaxes is None or not self._is_picker_ax(event.inaxes):
            return
        self._dragging = True
        self._picking_ax = event.inaxes
        self._old_xi = None
        self._old_yi = None
        self._process_event(event)

    def _on_motion(self, event):
        if not self._dragging:
            return
        if event.inaxes is None or not self._is_picker_ax(event.inaxes):
            self._dragging = False
            return
        self._process_event(event)

    def _on_release(self, event):
        self._dragging = False
        self._picking_ax = None
        self._old_xi = None
        self._old_yi = None

    def _on_reset(self, event):
        self.mask[:] = 0
        self._old_xi = None
        self._old_yi = None
        self._update_ifft()

    # ----------------------------------------------------------
    # マスク更新
    # ----------------------------------------------------------
    def _process_event(self, event):
        x, y = event.xdata, event.ydata
        if x is None or y is None:
            return

        xi = int(np.argmin(np.abs(self.fx - x)))
        yi = int(np.argmin(np.abs(self.fy - y)))

        draw_line_on_mask(self.mask, xi, yi, self._old_xi, self._old_yi)
        self._old_xi = xi
        self._old_yi = yi
        self._update_ifft()

    # ----------------------------------------------------------
    # IFFT 計算 & 表示更新
    # ----------------------------------------------------------
    def _update_ifft(self):
        # 再合成画像
        A = np.real(np.fft.ifft2(np.fft.ifftshift(self.mask * self.FFTedL)))
        A = A[:self.L.shape[0], :self.L.shape[1]]

        # 選択済みスペクトル
        picked_amp = self.mask * self.amp

        # 最後のグレーティング
        G = np.zeros_like(self.mask)
        if self._old_xi is not None:
            G[self._old_yi, self._old_xi] = 1
            G = np.real(np.fft.ifft2(np.fft.ifftshift(G * self.FFTedL)))
            G = G[:self.L.shape[0], :self.L.shape[1]]

        self.im_ifft.set_data(A)
        self.im_ifft.set_clim(vmin=A.min(), vmax=A.max())

        self.im_picker.set_data(picked_amp)
        self.im_picker.set_clim(vmin=picked_amp.min(), vmax=picked_amp.max())

        if self._old_xi is not None:
            self.im_grat.set_data(G)
            self.im_grat.set_clim(vmin=G.min(), vmax=G.max())

        self.fig.canvas.draw_idle()

In [ ]:
# -----------------------------------------------------------------
# 実行
# -----------------------------------------------------------------
plt.close('all')
demo = IFFTDemo(IMAGE_PATH)